# Lane 8: LOT-MoE - Google Colab Runner
**Project:** LOT-MoE (Lorentzian Optimal Transport Mixture of Experts)  
**Author:** Jaime Jarauta  
**Objective:** Eliminating MoE routing collapse with Zero Auxiliary Loss via Log-Domain Sinkhorn-Knopp iterations and Lorentz Hyperbolic Attention.

**Hardware Target:** Standard T4 GPU | **Max Runtime:** <= 15 minutes | **Cloud Credit Discipline:** Auto-Unassign On Completion or Error

In [ ]:
# Step 1: Credit Guardrail & Auto-Kill On Error
import sys, os, time, math
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython import get_ipython

# Safety Fallback: Automatically unassign Colab VM on ANY error to prevent idle burn
def sync_all_wandb_runs():
    """Flushes all offline or pending W&B runs to cloud before VM destruction."""
    import glob, subprocess
    candidates = glob.glob("/tmp/wandb/**/offline-run-*", recursive=True) + glob.glob("**/offline-run-*", recursive=True)
    if candidates:
        print(f"\n[W&B Guard] Detected {len(candidates)} offline run(s). Syncing to cloud before teardown...")
        for p in set(candidates):
            try:
                print(f"[W&B Guard] Syncing {p}...")
                subprocess.run(["wandb", "sync", p], check=False)
            except Exception as e:
                print(f"[W&B Warning] Sync failed for {p}: {e}")
    else:
        try:
            subprocess.run(["wandb", "sync", "--include-offline", "/tmp/wandb"], check=False)
            subprocess.run(["wandb", "sync", "--include-offline", "wandb"], check=False)
        except Exception:
            pass

def auto_kill_on_error(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    print("\n[Credit Guard Alert] Unhandled error caught! Flushing telemetry and unassigning Colab GPU...")
    try:
        sync_all_wandb_runs()
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), auto_kill_on_error)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# Step 1b: Clone Patched Repository & Install Dependencies
import os, sys, subprocess
if 'google.colab' in sys.modules:
    repo_dir = 'lot-moe'
    if not os.path.exists(repo_dir):
        print('Cloning lot-moe (branch: feature/initial-implementation)...')
        subprocess.run(['git', 'clone', '-b', 'feature/initial-implementation', 'https://github.com/jaimejarauta/lot-moe.git'], check=True)
    if os.path.exists(repo_dir):
        os.chdir(repo_dir)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', 'wandb', 'pytest'], check=True)
    if 'src' not in sys.path:
        sys.path.insert(0, 'src')
    print('Repository ready:', os.getcwd())


In [ ]:
# Step 1c: Secure Weights & Biases Authentication
# Securely reads from Colab Secrets (Key icon on left sidebar) or prompts interactively.
# API keys are NEVER hardcoded or committed into public git repositories.
import os
try:
    from google.colab import userdata
    key = userdata.get('WANDB_API_KEY')
    if key:
        os.environ['WANDB_API_KEY'] = key
        print('[W&B] Loaded WANDB_API_KEY from Google Colab Secrets.')
except Exception:
    pass

import wandb
if 'WANDB_API_KEY' not in os.environ:
    print('[W&B] Authenticate interactively below (or add WANDB_API_KEY to Colab Secrets):')
    wandb.login()
else:
    wandb.login(key=os.environ['WANDB_API_KEY'])

os.environ['WANDB_ENTITY'] = 'research-projekts'
print('[W&B] Authenticated successfully. Target entity: research-projekts')


In [ ]:
# Step 2: Self-Contained LOT-MoE Architecture Components
class LorentzMath:
    @staticmethod
    def spatial_to_lorentz(x_sp: torch.Tensor, c: float = 1.0) -> torch.Tensor:
        x_0 = torch.sqrt(1.0 / c + torch.sum(x_sp ** 2, dim=-1, keepdim=True))
        return torch.cat([x_0, x_sp], dim=-1)

class DualCurvatureAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, c: float = 1.0, beta: float = 1.0, hyperbolic_ratio: float = 0.5):
        super().__init__()
        self.d_model, self.num_heads = d_model, num_heads
        self.d_k = d_model // num_heads
        self.c, self.beta = c, beta
        self.num_hyp = int(num_heads * hyperbolic_ratio)
        self.num_euc = num_heads - self.num_hyp
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x: torch.Tensor, causal: bool = True) -> torch.Tensor:
        B, S, D = x.shape
        H, d_k = self.num_heads, self.d_k
        q = self.q_proj(x).view(B, S, H, d_k).transpose(1, 2)
        k = self.k_proj(x).view(B, S, H, d_k).transpose(1, 2)
        v = self.v_proj(x).view(B, S, H, d_k).transpose(1, 2)
        outputs = []
        if self.num_hyp > 0:
            qh, kh, vh = q[:, :self.num_hyp], k[:, :self.num_hyp], v[:, :self.num_hyp]
            q0 = torch.sqrt(1.0/self.c + (qh**2).sum(-1, keepdim=True))
            k0 = torch.sqrt(1.0/self.c + (kh**2).sum(-1, keepdim=True))
            mink = -torch.matmul(q0, k0.transpose(-2, -1)) + torch.matmul(qh, kh.transpose(-2, -1))
            dist_sq = (torch.acosh((-self.c * mink).clamp_min(1.0 + 1e-6)) ** 2) / self.c
            logits = -self.beta * dist_sq
            if causal:
                mask = torch.triu(torch.full((S, S), float('-inf'), device=x.device), diagonal=1)
                logits = logits + mask.unsqueeze(0).unsqueeze(0)
            outputs.append(torch.matmul(F.softmax(logits, dim=-1), vh))
        if self.num_euc > 0:
            qe, ke, ve = q[:, self.num_hyp:], k[:, self.num_hyp:], v[:, self.num_hyp:]
            logits = torch.matmul(qe, ke.transpose(-2, -1)) / math.sqrt(d_k)
            if causal:
                mask = torch.triu(torch.full((S, S), float('-inf'), device=x.device), diagonal=1)
                logits = logits + mask.unsqueeze(0).unsqueeze(0)
            outputs.append(torch.matmul(F.softmax(logits, dim=-1), ve))
        full = torch.cat(outputs, dim=1).transpose(1, 2).contiguous().view(B, S, D)
        return self.out_proj(full)

class LogSinkhornRouter(nn.Module):
    def __init__(self, d_model: int, num_experts: int, top_k: int = 2, epsilon: float = 0.05, iters: int = 15):
        super().__init__()
        self.num_experts, self.top_k, self.epsilon, self.iters = num_experts, top_k, epsilon, iters
        self.gate = nn.Linear(d_model, num_experts, bias=False)
    def forward(self, x_flat: torch.Tensor):
        N, E = x_flat.size(0), self.num_experts
        M = self.gate(x_flat) / self.epsilon
        log_r, log_c = -math.log(float(N)), -math.log(float(E))
        u, v = torch.zeros(N, device=x_flat.device), torch.zeros(E, device=x_flat.device)
        for _ in range(self.iters):
            u = log_r - torch.logsumexp(M + v.unsqueeze(0), dim=-1)
            v = log_c - torch.logsumexp(M + u.unsqueeze(1), dim=0)
        u = log_r - torch.logsumexp(M + v.unsqueeze(0), dim=-1)
        P = torch.exp(u.unsqueeze(1) + M + v.unsqueeze(0))
        topk_w, topk_idx = torch.topk(float(N) * P, self.top_k, dim=-1)
        topk_w = topk_w / topk_w.sum(dim=-1, keepdim=True).clamp_min(1e-7)
        return topk_w, topk_idx, P

class LOTMoEBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_experts: int, top_k: int = 2):
        super().__init__()
        self.num_experts, self.top_k = num_experts, top_k
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = DualCurvatureAttention(d_model, num_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.router = LogSinkhornRouter(d_model, num_experts, top_k=top_k)
        d_ffn = int(4 * d_model * 2 / 3)
        self.experts = nn.ModuleList([nn.Sequential(nn.Linear(d_model, d_ffn), nn.SiLU(), nn.Linear(d_ffn, d_model)) for _ in range(num_experts)])
    def forward(self, x: torch.Tensor):
        B, S, D = x.shape
        x = x + self.attn(self.norm1(x))
        h = self.norm2(x).view(B * S, D)
        topk_w, topk_idx, P = self.router(h)
        flat_out = torch.zeros_like(h)
        for e_idx in range(self.num_experts):
            for k in range(self.top_k):
                m = (topk_idx[:, k] == e_idx)
                if m.any(): flat_out[m] += topk_w[m, k:k+1] * self.experts[e_idx](h[m])
        return x + flat_out.view(B, S, D), P

print("LOT-MoE architecture compiled with Zero Auxiliary Loss guarantees.")

In [ ]:
# Step 3: Fast Multi-Expert Training & Zero Auxiliary Loss Benchmark
vocab_size, seq_len = 128, 64
num_experts, top_k = 8, 2

class FullModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 128)
        self.pos = nn.Embedding(seq_len, 128)
        self.layer = LOTMoEBlock(d_model=128, num_heads=4, num_experts=num_experts, top_k=top_k)
        self.head = nn.Linear(128, vocab_size, bias=False)
    def forward(self, x):
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        h, P = self.layer(self.emb(x) + self.pos(pos))
        return self.head(h), P

model = FullModel().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)

data = torch.randint(1, vocab_size, (1500, seq_len), device=device)
expert_counts = torch.zeros(num_experts, device=device)

print(f"Training LOT-MoE with E={num_experts} experts (Auxiliary Loss = 0.000)...\n")
t_start = time.time()
model.train()
for step in range(300):
    idx = torch.randint(0, 1500, (16,))
    batch = data[idx]
    logits, P = model(batch[:, :-1])
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), batch[:, 1:].reshape(-1))
    with torch.no_grad(): expert_counts += P.sum(dim=0)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if (step + 1) % 100 == 0 or step == 0:
        p_alloc = expert_counts.float() / expert_counts.sum()
        entropy = (-torch.sum(p_alloc * torch.log(p_alloc.clamp_min(1e-12))) / math.log(num_experts)).item()
        print(f"  Step {step+1:03d}/300 | Task Cross-Entropy: {loss.item():.4f} | Routing Entropy: {entropy*100:.2f}% | Aux Loss: 0.0")

p_final = expert_counts.float() / expert_counts.sum()
final_entropy = (-torch.sum(p_final * torch.log(p_final.clamp_min(1e-12))) / math.log(num_experts)).item()
print(f"\nTraining complete in {time.time() - t_start:.2f}s.")
print(f"Final Expert Allocation: {expert_counts.cpu().tolist()}")
print(f"Pre-Registered Gate 1 (Load Entropy >= 95% with L_aux=0): {'PASS' if final_entropy >= 0.95 else 'FAIL'} ({final_entropy*100:.2f}%)")

In [ ]:
# Step 3b: 16-Expert Causal Optimal Transport MoE Comparison
!pytest tests
!python scripts/benchmark_moe_comparison.py --num-experts 16 --top-k 2 --steps 300


In [ ]:
# Step Final: Credit Protection & Automatic Machine Teardown
# @title Auto-Kill Runtime on Completion to Save Colab Credits
AUTO_KILL_ON_FINISH = True # @param {type:"boolean"}

if AUTO_KILL_ON_FINISH:
    print("\n[W&B Guard] Flushing all local offline runs to cloud before VM teardown...")
    try:
        sync_all_wandb_runs()
    except Exception as e:
        print(f"(W&B pre-teardown sync notice: {e})")

    print("\n[Credit Guard] Run complete. Unassigning Colab GPU immediately to prevent idle credit burn...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"(Auto-kill skipped: not running inside interactive Colab: {e})")
